In [ ]:
Dataset Link= https://www.kaggle.com/datasets/shriyashjagtap/indian-personal-finance-and-spending-habits

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Download (first time) and cache locally (~250 MB)
model_name = "distilbert-base-uncased"



tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=7)


In [ ]:
import sys
import argparse

p = argparse.ArgumentParser()
p.add_argument("--mode", choices=["train", "demo"], default="demo")
p.add_argument("--model_dir", default="models/emotion_classifier_simple")
p.add_argument("--transactions", default="data/transactions_month.csv")

p.add_argument("--net_income", type=float, default=100000)

p.add_argument("--text", type=str, default="Feeling anxious about bills.")

# Only parse known args to avoid Jupyter's extra args
a, unknown = p.parse_known_args()  # Ignores extra args from Jupyter




In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load dataset
df = pd.read_csv("data
.csv")

# List of columns to include in text representation
columns_to_use = df.columns  # include all or specific ones

# Convert each row into a descriptive string
def row_to_text(row):
    text_parts = []
    for col in columns_to_use:
        text_parts.append(f"{col}: {row[col]}")
    return ", ".join(text_parts)

df['text'] = df.apply(row_to_text, axis=1)

# Generate labels based on savings
def assign_label(row):
    monthly_expenses = sum([row[col] for col in ['Rent', 'Loan_Repayment', 'Insurance',
                                                 'Groceries', 'Transport', 'Eating_Out',
                                                 'Entertainment', 'Utilities', 'Healthcare',
                                                 'Education', 'Miscellaneous']])
    savings = row['Income'] - monthly_expenses
    if savings < 0:
        return 'stressed'
    elif savings < 0.2 * row['Income']:
        return 'neutral'
    else:
        return 'confident'

df['label'] = df.apply(assign_label, axis=1)

# Split dataset
train_df, val_df = train_test_split(df[['text', 'label']], test_size=0.2, random_state=42)

# Save CSVs for Hugging Face dataset loader
train_df.to_csv("data_train.csv", index=False)
val_df.to_csv("data_val.csv", index=False)



In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset

# Configuration
class EmotionTrainConfig:
    def __init__(self):
        self.model_name = "distilbert-base-uncased"
        self.train_file = "data_train.csv"
        self.validation_file = "data_val.csv"
        self.text_col = "text"
        self.label_col = "label"
        self.max_length = 128
        self.batch_size = 16
        self.labels = ["stressed", "neutral", "confident"]
        self.output_dir = "./models/emotion_classifier_simple"

cfg = EmotionTrainConfig()



In [ ]:
def _build_label_maps(labels):
    label2id = {label: idx for idx, label in enumerate(labels)}
    id2label = {idx: label for label, idx in label2id.items()}
    return label2id, id2label

def _load_emotion_dataset(cfg, tokenizer):
    files = {"train": cfg.train_file, "validation": cfg.validation_file}
    raw = load_dataset("csv", data_files=files)

    l2i, i2l = _build_label_maps(cfg.labels)

    def enc(batch):
        tok = tokenizer(batch[cfg.text_col], truncation=True, max_length=cfg.max_length)
        tok["labels"] = [l2i[x] for x in batch[cfg.label_col]]
        return tok

    tokenized = raw.map(enc, batched=True, remove_columns=raw["train"].column_names)
    return tokenized, l2i, i2l



In [ ]:
def train_emotion_classifier(cfg):
    import os
    os.makedirs(cfg.output_dir, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
    tokenized, label2id, id2label = _load_emotion_dataset(cfg, tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(
        cfg.model_name,
        num_labels=len(cfg.labels),
        id2label=id2label,
        label2id=label2id
    )

    args = TrainingArguments(
        output_dir=cfg.output_dir,
        per_device_train_batch_size=cfg.batch_size,
        per_device_eval_batch_size=cfg.batch_size,
        num_train_epochs=3,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=50,
        save_total_limit=2,
        load_best_model_at_end=True,
        report_to=["none"]
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["validation"],
        tokenizer=tokenizer
    )

    trainer.train()
    model.save_pretrained(cfg.output_dir)
    tokenizer.save_pretrained(cfg.output_dir)



In [ ]:
class EmotionDetector:
    def __init__(self, model_dir):
        import os
        from transformers import AutoTokenizer, AutoModelForSequenceClassification
        if not os.path.exists(model_dir):
            raise FileNotFoundError(f"Model folder '{model_dir}' not found. Train the model first.")
        self.tokenizer = AutoTokenizer.from_pretrained(model_dir)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_dir)
        self.model.eval()

    def predict(self, text):
        from torch.nn.functional import softmax
        import torch
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
        outputs = self.model(**inputs)
        probs = softmax(outputs.logits, dim=1).detach().numpy()[0]
        label = self.model.config.id2label[probs.argmax()]
        return label

# Run training and demo
cfg = EmotionTrainConfig()
train_emotion_classifier(cfg)

det = EmotionDetector(cfg.output_dir)
sample = "Income: 60000, Age: 35, Dependents: 2, Rent: 15000, Groceries: 5000, Entertainment: 2000"
print("Predicted Emotion:", det.predict(sample))



In [ ]:
=====================ADVISORY=======================

In [ ]:
!pip install yfinance


In [ ]:
import sys
import yfinance as yf
import pandas as pd

# ===================== Data Loader =====================
class MarketData:
    def __init__(self, ticker="AAPL"):
        self.ticker = ticker

    def fetch_data(self, period="6mo", interval="1d"):
        data = yf.download(self.ticker, period=period, interval=interval)
        data.dropna(inplace=True)
        return data

# ===================== Indicator Calculator =====================
class IndicatorAnalyzer:
    def compute_indicators(self, df: pd.DataFrame):
        df["ma20"] = df["Close"].rolling(window=20).mean()
        df["ma50"] = df["Close"].rolling(window=50).mean()
        delta = df["Close"].diff()
        gain = (delta.where(delta > 0, 0)).rolling(14).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
        rs = gain / loss
        df["rsi"] = 100 - (100 / (1 + rs))
        return df

# ===================== Advisory Engine =====================
class AdvisoryEngine:
    def generate_nudges(self, ind: pd.DataFrame):
        notes = []
        ma20_last = ind["ma20"].iloc[-1]
        ma50_last = ind["ma50"].iloc[-1]
        rsi_last = ind["rsi"].iloc[-1]

        notes.append("📈 Bullish trend detected! Consider buying more."
                     if ma20_last > ma50_last
                     else "📉 Bearish trend detected! Consider reducing exposure.")

        if rsi_last > 70:
            notes.append("⚠️ RSI indicates overbought conditions.")
        elif rsi_last < 30:
            notes.append("💡 RSI indicates oversold conditions.")

        return notes

# ===================== Investment Executor =====================
class InvestmentExecutor:
    def simulate_trade(self, signal: str, price: float):
        return f"Simulated {signal} at price ${price:.2f}"

# ===================== Main Pipeline =====================
# def main(mode="demo"):
#     print(f"🚀 Running in {mode.upper()} mode")
#     data_loader = MarketData("AAPL")
#     df = data_loader.fetch_data()

#     analyzer = IndicatorAnalyzer()
#     indicators = analyzer.compute_indicators(df)

#     advisor = AdvisoryEngine()
#     nudges = advisor.generate_nudges(indicators)

#     executor = InvestmentExecutor()
#     last_price = df["Close"].iloc[-1]

#     # ✅ Output
#     print("\n💡 Investment Nudges:")
#     for n in nudges:
#         print("-", n)

#     print("\n💹 Sample Trade Execution:")
#     print(executor.simulate_trade("BUY", last_price))

# ===================== Jupyter/Terminal Safe Entry =====================
def main(mode="demo"):

    print(f"🚀 Running in {mode.upper()} mode")
    data_loader = MarketData("AAPL")
    df = data_loader.fetch_data()

    analyzer = IndicatorAnalyzer()
    indicators = analyzer.compute_indicators(df)

    advisor = AdvisoryEngine()
    nudges = advisor.generate_nudges(indicators)

    executor = InvestmentExecutor()
    last_price = float(df["Close"].iloc[-1])  # ✅ FIXED

    print("\n💡 Investment Nudges:")
    for n in nudges:
        print("-", n)

    print("\n💹 Sample Trade Execution:")
    print(executor.simulate_trade("BUY", last_price))

main("demo")





In [ ]:
# import numpy as np

# def label_trades(df):
#     """
#     Label 1 if the signal was correct (profitable after N days),
#     else 0.
#     """
#     N = 5  # look-ahead window (days)
#     df["future_return"] = df["Close"].shift(-N) / df["Close"] - 1
#     df["signal"] = np.where(df["ma20"] > df["ma50"], 1, 0)  # 1 = Buy, 0 = Sell
#     # Correct if Buy and price went up OR Sell and price went down
#     df["correct"] = np.where(
#         (df["signal"]==1) & (df["future_return"]>0)
#         | (df["signal"]==0) & (df["future_return"]<0),
#         1, 0
#     )
#     return df



In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

class MarketAnalyzer:
    def __init__(self, ticker="AAPL"):
        self.ticker = ticker

    def fetch_data(self, period="6mo", interval="1d"):
        df = yf.download(self.ticker, period=period, interval=interval, auto_adjust=True)
        if df.empty:
            raise ValueError(f"No data fetched for {self.ticker}")
        df = df.reset_index()
        # Pick price column robustly
        if "Close" in df.columns:
            self.price_col = "Close"
        elif "Adj Close" in df.columns:
            self.price_col = "Adj Close"
        elif "close" in df.columns:
            self.price_col = "close"
        else:
            raise KeyError("No Close or Adj Close column found in fetched data.")
        return df

    def compute_indicators(self, df):
        price = df[self.price_col]
        df["ma20"] = price.rolling(20).mean()
        df["ma50"] = price.rolling(50).mean()
        df["rsi"] = 100 - (100 / (1 + price.pct_change().rolling(14).mean()))
        return df

# Demo
analyzer = MarketAnalyzer("AAPL")
df = analyzer.fetch_data()
df = analyzer.compute_indicators(df)
print(df.tail())

